In [1]:
import pandas as pd
import altair as alt

# Load the data from the URL
url = 'https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/licenses_fall2022.csv'
df = pd.read_csv(url)

# Display basic information about the dataset
print(df.dtypes)
display(df.head())

_id                                          int64
License Type                                object
Description                                 object
License Number                              object
License Status                              object
Business                                    object
Title                                       object
First Name                                  object
Middle                                      object
Last Name                                   object
Prefix                                      object
Suffix                                      object
Business Name                               object
BusinessDBA                                 object
Original Issue Date                         object
Effective Date                              object
Expiration Date                             object
City                                        object
State                                       object
Zip                            

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN


In [2]:
display(df.describe())

,_id,Delegated Controlled Substance Schedule,Case Number
count,1.000000e+04,0.0,3.430000e+02
mean,7.787635e+05,NaN,2.005356e+09
std,2.901112e+05,NaN,9.249388e+06
min,2.792630e+05,NaN,1.982000e+09
25%,5.264690e+05,NaN,1.997526e+09
50%,7.806600e+05,NaN,2.004005e+09
75%,1.028364e+06,NaN,2.013504e+09
max,1.279042e+06,NaN,2.021011e+09


In [3]:
print(df.isnull().sum())

_id                                            0
License Type                                   0
Description                                    0
License Number                                60
License Status                                 0
Business                                       0
Title                                       9890
First Name                                   395
Middle                                      6378
Last Name                                    395
Prefix                                      9997
Suffix                                      9590
Business Name                                  0
BusinessDBA                                 9885
Original Issue Date                            5
Effective Date                               792
Expiration Date                              500
City                                          11
State                                          0
Zip                                           71
County              

In [4]:
# group for type and status
license_counts = df.groupby(['License Type', 'License Status']).size().reset_index(name='Count')

top_license_types = df['License Type'].value_counts().head(15).index
license_counts_filtered = license_counts[license_counts['License Type'].isin(top_license_types)]

print(license_counts)

             License Type License Status  Count
0               APPRAISAL        EXPIRED      5
1               ARCHITECT         ACTIVE      8
2               ARCHITECT       DECEASED      1
3               ARCHITECT       INACTIVE      5
4               ARCHITECT    NOT RENEWED      6
..                    ...            ...    ...
100  MASSAGE LICENSING BD       INACTIVE     12
101  MASSAGE LICENSING BD    NOT RENEWED     17
102         MEDICAL BOARD         ACTIVE      1
103         MEDICAL BOARD       INACTIVE      2
104         MEDICAL BOARD    NOT RENEWED      3

[105 rows x 3 columns]


This visualization displays the distribution of the top 15 most common license types in the dataset, showing how many licenses exist for each type and their current status (Active, Expired, Not Renewed, etc.). I used a horizontal bar chart format with the License Type encoded as a nominal variable on the y-axis, Count encoded as a quantitative variable on the x-axis, and License Status encoded as a nominal variable using color. The bars are sorted in descending order by total count to make it easy to identify which license types are most prevalent. For the color encoding, I chose the category20 color scheme for License Status because it provides clear visual distinction between the different status categories, making it easy to compare status distributions across license types. In terms of data transformations, I grouped the original dataset by both License Type and License Status to aggregate the counts, then filtered the results to include only the top 15 license types by total count to prevent overcrowding and maintain readability.

In [5]:
# Horizontal bar
chart1 = alt.Chart(license_counts_filtered).mark_bar().encode(
    y=alt.Y('License Type:N', sort='-x', title='License Type'),
    x=alt.X('Count:Q', title='Number of Licenses'),
    color=alt.Color('License Status:N', title='License Status', scale=alt.Scale(scheme='category20')),
    tooltip=[
        alt.Tooltip('License Type:N', title='License Type'),
        alt.Tooltip('License Status:N', title='Status'),
        alt.Tooltip('Count:Q', title='Count', format=',')
    ]
).properties(
    title='Distribution of License Types by Status',
    width=600,
    height=400
).interactive()

chart1

alt.Chart(...)

In [9]:
df_no_il = df[df['State'] != 'IL'] #dont include illinois as it has too many entries and ruins the scale
top_10_states = df_no_il['State'].value_counts().head(10).index
state_status = df_no_il[df_no_il['State'].isin(top_10_states)].groupby(['State', 'License Status']).size().reset_index(name='Count')

   State            License Status  Count
0     AZ                    ACTIVE      3
1     AZ                  INACTIVE      5
2     AZ               NOT RENEWED     21
3     CA                    ACTIVE      7
4     CA                 CANCELLED      2
5     CA                  INACTIVE     10
6     CA               NOT RENEWED     62
7     CA  TERMINATED CARD RETURNED      3
8     FL                    ACTIVE      8
9     FL                 CANCELLED      2
10    FL                   EXPIRED      3
11    FL                  INACTIVE     20
12    FL               NOT RENEWED     53
13    GA                    ACTIVE      4
14    GA       CHANGE OF OWNERSHIP      1
15    GA                    CLOSED      2
16    GA                  INACTIVE      3
17    GA               NOT RENEWED     17
18    IA                    ACTIVE     13
19    IA                 CANCELLED      1


In [12]:
status_dropdown = alt.binding_select(
    options=[None] + sorted(df['License Status'].unique().tolist()),
    labels=['All Statuses'] + sorted(df['License Status'].unique().tolist()),
    name='Select License Status: '
)

status_select = alt.selection_point(
    fields=['License Status'],
    bind=status_dropdown,
    value=None
)

chart2 = alt.Chart(state_status).mark_bar().encode(
    x=alt.X('State:N', 
            title='State',
            sort='-y'),
    y=alt.Y('Count:Q', title='Number of Licenses'),
    color=alt.Color('License Status:N',
                    title='License Status',
                    scale=alt.Scale(scheme='category10')),
    tooltip=[
        alt.Tooltip('State:N', title='State'),
        alt.Tooltip('License Status:N', title='Status'),
        alt.Tooltip('Count:Q', title='Count', format=',')
    ]
).transform_filter(
    status_select
).add_params(
    status_select
).properties(
    title='License Status Distribution Across Top 10 States (Excluding IL)',
    width=600,
    height=400
)

chart2

alt.Chart(...)

This visualization shows the distribution of license statuses across the top 10 states by license count, excluding Illinois to prevent scale distortion since Illinois had an overwhelming number of licenses compared to other states. I used a grouped bar chart with State encoded as a nominal variable on the x-axis (sorted by total count in descending order), Count encoded as a quantitative variable on the y-axis, and License Status encoded as a nominal variable using color with an xOffset to group bars side by side. The color encoding uses the category10 color scheme, as similarly to category20, it provides clear visual distinction between the different status categories, making it easy to compare status distributions across license statuses. For data transformations, I first filtered out all records where the state was Illinois, then identified the top 10 states by total license count, and finally grouped the filtered dataset by both State and License Status to create the aggregated counts used in the visualization. The grouped bar format allows for easy comparison both within states (comparing different statuses) and across states (comparing the same status).

The first visualization uses the .interactive() method which provides basic pan and zoom functionality. The second visualization includes a dropdown selector that filters the chart by License Status, which goes beyond basic pan/zoom and represents custom interactivity as required by the assignment. This dropdown makes the visualization clearer by letting users focus on one license status at a time, making state-to-state comparisons easier without visual clutter from other categories.

In [8]:
chart1.save('hw5_chart1.json')
chart2.save('hw5_chart2.json')